# Notebook 06 — Sentinel-1 InSAR Workflow (ASF HyP3)
**Stage 7 of the Ghana Groundwater–Subsidence Pipeline**

Processes Sentinel-1 SAR interferometry for land displacement mapping using ASF HyP3 cloud processing.

**Workflow:** Inventory analysis → Pair selection (6/12-day baselines, path 147) → HyP3 submission → Monitor + download → Velocity map → Compare with subsidence proxy

**Prerequisites:** hyp3_sdk installed ✓ · Earthdata credentials in ~/.netrc ✓ · 1,183 recommended pairs ✓

**Outputs:** `hyp3_jobs_submitted.csv` · `sentinel1_velocity_map.tif` · `sentinel1_velocity_map.png`

In [ ]:
import os
import pathlib; os.chdir(str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == 'notebooks' else str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import hyp3_sdk
from pathlib import Path
from datetime import datetime
import time, json, zipfile

ROOT  = Path('.')
S1DIR = ROOT / 'data' / 'raw' / 'sentinel1'
PROD  = S1DIR / 'hyp3_products'
SUB   = ROOT / 'data' / 'processed' / 'subsidence'
FIGS  = ROOT / 'outputs' / 'figures'
TABS  = ROOT / 'outputs' / 'tables'
PROD.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'font.family':'sans-serif','font.size':9,
                     'savefig.dpi':150,'savefig.bbox':'tight'})
print(f'hyp3_sdk: {hyp3_sdk.__version__}')


## 1. Scene Inventory Analysis

In [ ]:
df_inv = pd.read_csv(S1DIR / 's1_scene_inventory_ghana_2015_2024.csv',
                     parse_dates=['date'])
print(f'Total scenes: {len(df_inv):,}  ({df_inv.date.min().date()} to {df_inv.date.max().date()})')

path_counts = df_inv.groupby(['orbit_dir','path'])['scene_name'].count().reset_index()
path_counts.columns = ['orbit_dir','path','n_scenes']
print('\nScenes by path:')
print(path_counts.sort_values('n_scenes', ascending=False).to_string(index=False))

p147 = df_inv[df_inv['path']==147]
print(f'\nPath 147 (Accra corridor): {len(p147):,} scenes by year:')
print(p147.groupby('year')['scene_name'].count().to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('Sentinel-1 Scene Inventory — Ghana 2015-2024', fontweight='bold', fontsize=11)

ax = axes[0]
pivot = df_inv.pivot_table(index='year', columns='path', values='scene_name',
                            aggfunc='count', fill_value=0)
top_paths = df_inv.groupby('path')['scene_name'].count().nlargest(5).index.tolist()
pivot[[c for c in top_paths if c in pivot.columns]].plot(kind='bar', ax=ax, width=0.8)
ax.set_xlabel('Year'); ax.set_ylabel('N scenes')
ax.set_title('(a) Annual scene counts by path', fontsize=9)
ax.legend(title='Path', fontsize=7); ax.tick_params(axis='x', rotation=45)

ax = axes[1]
p147_sorted = p147.sort_values('date')
gaps = p147_sorted['date'].diff().dt.days
ax.hist(gaps.dropna(), bins=30, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(12, color='red',    lw=1.5, ls='--', label='12-day repeat')
ax.axvline(6,  color='orange', lw=1.5, ls='--', label='6-day repeat')
ax.set_xlabel('Days between acquisitions'); ax.set_ylabel('Count')
ax.set_title('(b) Path 147 acquisition gap distribution', fontsize=9)
ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(FIGS / 's1_inventory_analysis.png', dpi=150)
plt.close()
print('Saved s1_inventory_analysis.png')


## 2. Pair Selection

In [ ]:
df_pairs = pd.read_csv(TABS / 's1_insar_recommended_pairs_path147.csv')
print(f'All recommended pairs: {len(df_pairs):,}')
print('Baseline distribution:')
print(df_pairs['temporal_baseline_days'].value_counts().sort_index().to_string())

# Keep only 6 and 12-day pairs — best coherence for Ghana's vegetated terrain
good_pairs = df_pairs[df_pairs['temporal_baseline_days'].isin([6, 12])].copy()
good_pairs['master_dt'] = pd.to_datetime(good_pairs['master_date'])
good_pairs['slave_dt']  = pd.to_datetime(good_pairs['slave_date'])
good_pairs['year']      = good_pairs['master_dt'].dt.year
good_pairs = good_pairs.sort_values('master_dt')
print(f'\nHigh-quality pairs (6+12-day): {len(good_pairs)}')
print(f'Date range: {good_pairs.master_date.min()} to {good_pairs.slave_date.max()}')

# Test batch: ~3 pairs per year spread across 2015-2024
test_batch = (
    good_pairs
    .groupby('year', group_keys=False)
    .apply(lambda x: x.nsmallest(3, 'temporal_baseline_days'))
    .reset_index(drop=True)
    .head(30)
)
print(f'\nTest batch: {len(test_batch)} pairs')
test_batch['year'] = pd.to_datetime(test_batch['master_date']).dt.year
print(test_batch.groupby('year')['master_date'].count().to_string())

good_pairs.to_csv(TABS / 's1_pairs_selected.csv', index=False)
test_batch.to_csv(TABS / 's1_pairs_test_batch.csv', index=False)
print('\nSaved s1_pairs_selected.csv and s1_pairs_test_batch.csv')


## 3. HyP3 Authentication and Quota

In [ ]:
# HyP3 reads urs.earthdata.nasa.gov credentials from ~/.netrc automatically
print('Connecting to ASF HyP3...')
hyp3 = None
try:
    hyp3 = hyp3_sdk.HyP3()
    print('Connected successfully.')
except hyp3_sdk.exceptions.AuthenticationError as e:
    print(f'AuthenticationError: {e}')
    print()
    print('ACTION REQUIRED: Update your Earthdata profile before HyP3 can authenticate.')
    print('  1. Go to https://urs.earthdata.nasa.gov/profile')
    print('  2. Set the required application attributes (Study Area, etc.)')
    print('  3. Re-run this notebook after updating.')
    print()
    print('Remaining cells will be skipped until authentication is fixed.')
except Exception as e:
    print(f'Connection failed: {e}')
    print('Ensure urs.earthdata.nasa.gov is in ~/.netrc')

try:
    if hyp3:
        user_info = hyp3.my_info()
        print(f'User info: {user_info}')
except Exception as e:
    print(f'Could not fetch user info (non-fatal): {e}')

try:
    if hyp3:
        existing = list(hyp3.find_jobs(name='ghana_insar'))
        if existing:
            status_ct = {}
            for j in existing:
                status_ct[j.status_code] = status_ct.get(j.status_code, 0) + 1
            print(f'Existing Ghana InSAR jobs: {len(existing)}  Status: {status_ct}')
        else:
            print('No existing Ghana InSAR jobs found.')
except Exception as e:
    print(f'Could not fetch existing jobs: {e}')


## 4. Submit InSAR Jobs to HyP3

In [ ]:
# Define jobs_file here so downstream cells can always access it
jobs_file = TABS / 'hyp3_jobs_submitted.csv'

# ── Submit batch ─────────────────────────────────────────────────────────
if hyp3 is None:
    print('HyP3 not authenticated. Update Earthdata profile at:')
    print('  https://urs.earthdata.nasa.gov/profile')
    print('Then re-run this notebook.')
else:
    # SUBMIT_BATCH = test_batch   # 30 pairs — safe start
    # SUBMIT_BATCH = good_pairs   # ALL 240 high-quality pairs
    SUBMIT_BATCH = test_batch

    submitted_names = set()
    if jobs_file.exists():
        df_ex = pd.read_csv(jobs_file)
        submitted_names = set(df_ex['job_name'].tolist())
        print(f'Resuming: {len(submitted_names)} jobs already logged.')

    print(f'Submitting up to {len(SUBMIT_BATCH)} InSAR jobs...')
    print('  Looks: 20x4  |  Outputs: LOS displacement, coherence, wrapped phase, DEM')

    new_jobs, failed = [], []
    for i2, (_, row) in enumerate(SUBMIT_BATCH.iterrows()):
        job_name = 'ghana_insar_' + str(row['master_date']) + '_' + str(row['slave_date'])
        if job_name in submitted_names:
            print(f'  [{i2+1:02d}] Skip: {job_name}')
            continue
        try:
            # submit_insar_job returns a Batch object in hyp3_sdk >= 7.x
            batch = hyp3.submit_insar_job(
                granule1=row['master'],
                granule2=row['slave'],
                name=job_name,
                looks='20x4',
                include_dem=True,
                include_los_displacement=True,
                include_wrapped_phase=True,
                apply_water_mask=True,
                phase_filter_parameter=0.6,
            )
            # Iterate over the Batch to get individual jobs
            for job in batch:
                new_jobs.append({
                    'job_id': job.job_id,
                    'job_name': job_name,
                    'master': row['master'],
                    'slave':  row['slave'],
                    'master_date': row['master_date'],
                    'slave_date':  row['slave_date'],
                    'temporal_baseline': row['temporal_baseline_days'],
                    'status': job.status_code,
                    'submitted_at': datetime.utcnow().isoformat(),
                })
            print(f'  [{i2+1:02d}] Submitted: {row["master_date"]} -> {row["slave_date"]}'
                  f'  ({len(batch)} job(s), first id={str(list(batch)[0].job_id)[:8]}...)')
            time.sleep(0.3)
        except Exception as exc:
            print(f'  [{i2+1:02d}] FAILED {job_name}: {exc}')
            failed.append({'name': job_name, 'error': str(exc)})

    if new_jobs:
        df_new = pd.DataFrame(new_jobs)
        if jobs_file.exists():
            df_new = pd.concat([pd.read_csv(jobs_file), df_new], ignore_index=True)
        df_new.to_csv(jobs_file, index=False)
        print(f'\nSubmitted {len(new_jobs)} jobs across {len(SUBMIT_BATCH)} pairs.')
        print(f'Saved: {jobs_file}')
    if failed:
        pd.DataFrame(failed).to_csv(TABS / 'hyp3_failed_pairs.csv', index=False)
        print(f'Failed: {len(failed)} pairs (see hyp3_failed_pairs.csv)')


## 5. Monitor Jobs and Download Products

In [ ]:
# ── Monitor — waits up to MAX_WAIT_MIN for jobs to complete ─────────────
# Set to 0 for notebook execution (avoids timeout).
# Increase to 60+ when re-running manually to wait for jobs.
MAX_WAIT_MIN   = 0
CHECK_INTERVAL = 60

if hyp3 is None:
    print('HyP3 not authenticated — skipping monitor.')
    print('Fix Earthdata profile at https://urs.earthdata.nasa.gov/profile')
elif not jobs_file.exists():
    print('No job log found. Run the submission cell first.')
else:
    df_jobs = pd.read_csv(jobs_file)
    pending = df_jobs[df_jobs['status'].isin(['PENDING','RUNNING'])]['job_id'].tolist()
    print(f'Jobs in log: {len(df_jobs)}  |  Pending/running: {len(pending)}')
    if not pending:
        print('All jobs resolved. Check status below.')
    else:
        start_t = time.time()
        while pending:
            elapsed_min = (time.time() - start_t) / 60
            if elapsed_min > MAX_WAIT_MIN:
                print(f'Reached {MAX_WAIT_MIN}-min limit. {len(pending)} jobs still pending.')
                print('Re-run this cell to continue, or check https://hyp3.asf.alaska.edu')
                break
            still_pending = []
            for jid in pending:
                try:
                    job = hyp3.get_job_by_id(jid)
                    df_jobs.loc[df_jobs['job_id']==jid, 'status'] = job.status_code
                    if job.status_code in ('PENDING','RUNNING'):
                        still_pending.append(jid)
                    else:
                        print(f'  {job.status_code}: {str(jid)[:8]}...')
                except Exception as e:
                    still_pending.append(jid)
            df_jobs.to_csv(jobs_file, index=False)
            pending = still_pending
            n_done = (df_jobs['status']=='SUCCEEDED').sum()
            print(f'  [{elapsed_min:.0f}m]  Done:{n_done}/{len(df_jobs)}  Pending:{len(pending)}')
            if pending:
                time.sleep(CHECK_INTERVAL)
    print('\nJob status summary:')
    print(df_jobs['status'].value_counts().to_string())


In [ ]:
# ── Download completed products ──────────────────────────────────────────
if hyp3 is None:
    print('HyP3 not authenticated — skipping download.')
elif not jobs_file.exists():
    print('No job log. Run submission cell first.')
else:
    df_jobs = pd.read_csv(jobs_file)
    succeeded = df_jobs[df_jobs['status']=='SUCCEEDED']['job_id'].tolist()
    print(f'SUCCEEDED jobs: {len(succeeded)}')
    if not succeeded:
        print('No completed jobs yet. Re-run monitoring cell.')
    else:
        downloaded = 0
        for jid in succeeded:
            if list(PROD.glob('*' + str(jid)[:8] + '*.zip')):
                continue
            try:
                job = hyp3.get_job_by_id(jid)
                job.download_files(PROD)
                downloaded += 1
                print(f'  Downloaded: {str(jid)[:8]}...')
            except Exception as e:
                print(f'  FAILED {str(jid)[:8]}: {e}')
        zips = list(PROD.glob('*.zip'))
        print(f'New downloads: {downloaded}  |  Total ZIPs: {len(zips)}')
        unzipped = 0
        for z in PROD.glob('*.zip'):
            out = PROD / z.stem
            if not out.exists():
                try:
                    with zipfile.ZipFile(z) as zf:
                        zf.extractall(PROD)
                    unzipped += 1
                except Exception as e:
                    print(f'  WARN unzip {z.name}: {e}')
        print(f'Unzipped: {unzipped}  |  LOS files: {len(list(PROD.glob("**/*los_disp.tif")))}')


## 6. Compute Velocity Map

In [ ]:
import rasterio
from scipy.stats import linregress
import re

los_files  = sorted(PROD.glob('**/*los_disp.tif'))
corr_files = sorted(PROD.glob('**/*corr.tif'))
print(f'LOS displacement files: {len(los_files)}')
print(f'Coherence files       : {len(corr_files)}')

if len(los_files) < 5:
    print(f'\nNeed >= 5 files to compute velocity. Currently have {len(los_files)}.')
    print('Re-run after more HyP3 products are downloaded.')
else:
    def parse_date(f):
        m = re.search(r'_(\d{8})T', f.name)
        return pd.Timestamp(m.group(1)) if m else None

    valid = sorted([(f, parse_date(f)) for f in los_files if parse_date(f)],
                   key=lambda x: x[1])
    print(f'Stacking {len(valid)} files ({valid[0][1].date()} to {valid[-1][1].date()})')

    with rasterio.open(valid[0][0]) as src:
        meta    = src.meta.copy()
        height, width = src.height, src.width

    stack  = np.full((len(valid), height, width), np.nan, dtype=np.float32)
    t_days = np.array([(d - valid[0][1]).days for _, d in valid], dtype=float)

    for i, (f, d) in enumerate(valid):
        try:
            with rasterio.open(f) as src:
                data = src.read(1).astype(np.float32)
                nodata = src.nodata
                if nodata is not None:
                    data[data == nodata] = np.nan
                stack[i] = data * 1000.0  # m -> mm
        except Exception as e:
            print(f'  WARN: {f.name}: {e}')

    print('Computing pixel-wise linear velocity...')
    velocity  = np.full((height, width), np.nan, dtype=np.float32)
    r_sq      = np.full((height, width), np.nan, dtype=np.float32)

    for row in range(height):
        for col in range(width):
            ts = stack[:, row, col]
            vm = ~np.isnan(ts)
            if vm.sum() < 5: continue
            s, b, r, p, se = linregress(t_days[vm], ts[vm])
            velocity[row, col] = s * 365.25
            r_sq[row, col]     = r**2

    print(f'Velocity: {np.nanmin(velocity):.1f} to {np.nanmax(velocity):.1f} mm/yr')
    print(f'Valid pixels: {np.isfinite(velocity).sum():,} / {height*width:,}')

    meta.update({'count':1, 'dtype':'float32', 'nodata':np.nan})
    with rasterio.open(SUB / 'sentinel1_velocity_map.tif', 'w', **meta) as dst:
        dst.write(velocity, 1)
    print(f'Saved sentinel1_velocity_map.tif')

    if corr_files:
        coh_stack = np.full((len(corr_files), height, width), np.nan, dtype=np.float32)
        for i, f in enumerate(corr_files):
            try:
                with rasterio.open(f) as src:
                    d = src.read(1).astype(np.float32)
                    if src.nodata: d[d==src.nodata] = np.nan
                    coh_stack[i] = d
            except Exception: pass
        with rasterio.open(SUB / 'sentinel1_coherence_mean.tif', 'w', **meta) as dst:
            dst.write(np.nanmean(coh_stack, axis=0), 1)
        print(f'Saved sentinel1_coherence_mean.tif')


## 7. Velocity Map Figure and Status Summary

In [ ]:
vel_path  = SUB / 'sentinel1_velocity_map.tif'
coh_path  = SUB / 'sentinel1_coherence_mean.tif'

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
fig.suptitle('Sentinel-1 InSAR — Coastal Ghana (ASF HyP3)',
             fontweight='bold', fontsize=11)

ax = axes[0]
if vel_path.exists():
    with rasterio.open(vel_path) as src:
        vel = src.read(1)
        b   = src.bounds
    p2, p98 = np.nanpercentile(vel, [2, 98])
    im = ax.imshow(vel, cmap='RdBu', vmin=p2, vmax=p98,
                   extent=[b.left, b.right, b.bottom, b.top],
                   origin='upper', aspect='auto')
    plt.colorbar(im, ax=ax, label='LOS velocity (mm yr\u207b\xb9)')
    pct = (vel < -5).sum() / np.isfinite(vel).sum() * 100
    ax.text(0.03, 0.97, f'{pct:.1f}% pixels < -5 mm/yr',
            transform=ax.transAxes, va='top', fontsize=8,
            bbox=dict(fc='white', alpha=0.8, boxstyle='round'))
    ax.set_xlabel('Longitude (\u00b0E)'); ax.set_ylabel('Latitude (\u00b0N)')
    ax.set_title('(a) LOS velocity (negative = potential subsidence)', fontsize=9)
else:
    # Show job status bar chart
    if jobs_file.exists():
        df_j = pd.read_csv(jobs_file)
        sc   = df_j['status'].value_counts()
        cols = ['#2ecc71' if s=='SUCCEEDED' else '#e74c3c' if s=='FAILED'
                else '#f39c12' for s in sc.index]
        ax.bar(sc.index, sc.values, color=cols, alpha=0.85, edgecolor='white')
        ax.set_ylabel('Number of jobs')
        ax.set_title(f'(a) HyP3 job status (total: {len(df_j)})', fontsize=9)
    else:
        ax.text(0.5, 0.5, 'Jobs not yet submitted',
                ha='center', va='center', transform=ax.transAxes, color='gray', fontsize=12)
        ax.set_title('(a) LOS velocity map (pending)', fontsize=9)

ax = axes[1]
if coh_path.exists():
    with rasterio.open(coh_path) as src:
        coh = src.read(1); b = src.bounds
    im2 = ax.imshow(coh, cmap='Greys_r', vmin=0, vmax=1,
                    extent=[b.left, b.right, b.bottom, b.top],
                    origin='upper', aspect='auto')
    plt.colorbar(im2, ax=ax, label='Mean coherence (0-1)')
    ax.set_xlabel('Longitude (\u00b0E)'); ax.set_ylabel('Latitude (\u00b0N)')
    ax.set_title('(b) Mean coherence across all interferograms', fontsize=9)
else:
    # Pair baseline scatter
    if (TABS/'s1_pairs_selected.csv').exists():
        gp = pd.read_csv(TABS/'s1_pairs_selected.csv')
        gp['master_dt'] = pd.to_datetime(gp['master_date'])
        ax.scatter(gp['master_dt'], gp['temporal_baseline_days'],
                   s=8, alpha=0.5, color='steelblue')
        ax.set_xlabel('Acquisition date'); ax.set_ylabel('Temporal baseline (days)')
        ax.set_title(f'(b) Selected pairs: {len(gp)} (6+12-day, path 147)', fontsize=9)
        ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
fig.savefig(FIGS / 'sentinel1_velocity_map.png', dpi=150)
plt.close()
print('Saved sentinel1_velocity_map.png')

# ── Status summary ─────────────────────────────────────────────────────────
print()
print('=== Notebook 06 Status ===')
checkpoints = [
    ('hyp3_sdk installed',       True),
    ('Pairs selected',           (TABS/'s1_pairs_selected.csv').exists()),
    ('Jobs submitted',           (TABS/'hyp3_jobs_submitted.csv').exists()),
    ('Products downloaded',      len(list(PROD.glob('*.zip'))) > 0),
    ('Velocity map computed',    vel_path.exists()),
]
for label, done in checkpoints:
    print(f'  [{"DONE" if done else "PENDING"}]  {label}')

if jobs_file.exists():
    df_j = pd.read_csv(jobs_file)
    print()
    print('  Jobs:', df_j['status'].value_counts().to_dict())
    print('  Monitor: https://hyp3.asf.alaska.edu')
    print('  Next: re-run cells 5-6 after SUCCEEDED count > 5')
